# M09. 멀티인덱스 (다층 헤더 처리)

> 📌 **언제 필요한가**  
> 한국 공공데이터(KOSIS 등)에서 헤더가 여러 줄에 걸쳐 있을 때.  
> 예: 1행 "성별", 2행 "남/여", 3행 "분석대상자수 (명)" 처럼 헤더가 3~4행짜리.

## 이 모듈에서 배울 것

- 다층 헤더가 뭔지, 왜 만나는지
- `pd.read_csv(header=[0,1,2,3])` 한 줄로 다층 헤더 읽기
- `rename`으로 컬럼명 정리 (단위 제거 · 불필요한 레벨 제거 · 레벨 이름 붙이기)
- 멀티인덱스 데이터 접근 (`df['전체']['남학생']['인지율']`)

## 💡 주의

이 모듈에서는 청소년 스트레스 인지율 데이터를 사용해요.  
**정제 기법 학습이 목적**이고 본 분석은 다루지 않아요. 분석 주제는 본인 프로젝트에서 자유롭게.

---

## 📥 데이터 준비

> 이 모듈은 아래 파일이 필요해요. **저장소에는 동봉돼 있지 않으니** 먼저 받아서 두세요.
> 받는 곳 링크를 누르면 바로 받으러 갈 수 있어요.

- `스트레스_인지율.csv` — 인코딩 `utf-8` — [KOSIS에서 받기](https://kosis.kr/statisticsList/statisticsListIndex.do?menuId=M_01_01&vwcd=MT_ZTITLE&parmTabId=M_01_01&parentId=F.1;F_55.2;117_11758_011.3;&outLink=Y#117_11758_011.3) ("스트레스 인지율" 선택)
- `우울감_경험률.csv` — 인코딩 `utf-8` — [KOSIS에서 받기](https://kosis.kr/statisticsList/statisticsListIndex.do?menuId=M_01_01&vwcd=MT_ZTITLE&parmTabId=M_01_01&parentId=F.1;F_55.2;117_11758_011.3;&outLink=Y#117_11758_011.3) ("우울감 경험률" 선택)

두는 곳 — **로컬 Jupyter**: 이 노트북과 같은 폴더 / **Colab**: `/content/`에 업로드.  
컬럼 설명·함정 등 자세한 내용은 [`data/README.md`](data/README.md) 참고.

---

## 1. 데이터 받아보기 — 어, 헤더가 이상한데?

먼저 평범하게 읽어봅시다.


In [ ]:
import pandas as pd

stress = pd.read_csv('data/스트레스_인지율.csv')
print(f"shape: {stress.shape}")
stress.head()


**눈으로 보면 이상한 점 두 가지**:

1. **컬럼명이 "전체", "전체.1", "전체.2"...** 처럼 의미 없음 (pandas가 중복 자동 번호)
2. **데이터 처음 3행이 사실 데이터가 아니라 헤더 정보** ("소계", "남학생", "인지율 (%)" 등)

진짜 숫자 데이터는 4번째 행(2005년)부터 시작해요. **헤더가 4행짜리**라는 뜻이에요.

```
실제 CSV 구조:
L0: 전체   전체    전체     ...  학년별  ...   ← 큰 분류
L1: 소계   소계    소계     ...  중1     ...   ← 세부
L2: 전체   전체    전체     ...  남학생  ...   ← 성별
L3: 분석대상자수(명) 인지율(%) 표준오차 ...      ← 측정 변수
L4: 2005   58224   45.6    0.3  ...           ← 진짜 데이터 시작
L5: 2006   ...
```

> ⚠️ **레벨 순서는 파일마다 달라요.** 이 파일은 `… → 성별(L2) → 측정변수(L3)` 순서예요.
> 다른 KOSIS 파일은 순서가 반대일 수도 있으니, **접근하기 전에 `head()`로 실제 순서를 꼭 확인**하세요.

## 2. `header=` 옵션으로 읽기 (한 줄)

가장 간단한 방법. `read_csv`에 `header=` 옵션으로 헤더 행 번호들을 리스트로 주면 멀티인덱스로 한 번에 읽어줘요.

In [ ]:
# 헤더가 4행 (L0, L1, L2, L3) → header=[0, 1, 2, 3]
# 시점 컬럼을 인덱스로 → index_col=0
stress = pd.read_csv('data/스트레스_인지율.csv', header=[0, 1, 2, 3], index_col=0)
print(f"shape: {stress.shape}")
print(f"\n컬럼 (앞 6개):")
for col in stress.columns[:6]:
    print(f"  {col}")

In [ ]:
# 첫 3행 보기
stress.head(3).iloc[:, :6]

**잘 읽혔어요!** 컬럼이 **4중 멀티인덱스** (튜플 4개)로 들어갔어요: `(분류, 세부, 성별, 측정변수)`.

읽기는 한 줄로 끝났지만, 이대로 쓰기엔 두 가지가 거슬려요:
- 단위가 컬럼명에 그대로 박힘 (`분석대상자수 (명)`, `인지율 (%)`)
- 안 쓰는 레벨(`분류`)까지 4중이라 접근이 번거로움

다음 섹션에서 `rename`으로 정리해볼게요.

## 3. 컬럼명 정리 — `rename` 후처리

읽는 방법은 그대로 두고, 받은 멀티인덱스의 **컬럼명만** 손봐요. 핵심은 `rename`의 `level=` 인자 — "어느 층만 바꿀지" 지정할 수 있어서, 구조를 다시 조립하지 않고 원하는 레벨만 고쳐요.

In [ ]:
# 방법 A로 받은 stress(4중)를 정리 → 3중 + 단위 제거 + 레벨 이름
stress = (
    stress
      .droplevel(0, axis=1)                              # 안 쓰는 '분류'(L0) 레벨 제거 → 3중
      .rename(columns={'소계': '전체'}, level=0)          # '세부' 레벨: 소계 → 전체 (의미 명확화)
      .rename(columns={'분석대상자수 (명)': '분석대상자수',  # '측정변수' 레벨: 단위 제거 (값 직접 매핑)
                       '인지율 (%)': '인지율'}, level=2)
      .rename_axis(columns=['세부', '성별', '측정변수'])    # 레벨에 이름 붙이기
)
print(f"shape: {stress.shape}")
print(f"레벨 이름: {stress.columns.names}")
print("컬럼 (앞 6개):")
for col in stress.columns[:6]:
    print(f"  {col}")

In [ ]:
# 첫 3행 보기
stress.head(3).iloc[:, :6]

**정리 완료!** **3중 멀티인덱스**(`세부 / 성별 / 측정변수`)에 단위도 빠지고 레벨 이름까지 붙었어요.

| 단계 | 한 일 |
|---|---|
| `droplevel(0, axis=1)` | 안 쓰는 `분류` 레벨 제거 (4중 → 3중) |
| `rename(columns={'소계':'전체'}, level=0)` | 특정 값만 교체 |
| `rename(columns={'인지율 (%)':'인지율', ...}, level=2)` | 한 레벨에서 단위 제거 (값을 직접 매핑) |
| `rename_axis(columns=[...])` | 레벨에 이름 부여 → `xs(level=...)` 가 편해짐 |

> 💡 **핵심은 `rename`의 `level=`** — 멀티인덱스 전체를 다시 조립(`from_arrays`)하지 않고, 필요한 레벨만 콕 집어 고칠 수 있어요. 바꿀 값이 몇 개 안 되면 `{옛이름: 새이름}` dict가 가장 명확해요.

## 4. 멀티인덱스 데이터 접근

이제 정리한 `stress`에서 데이터를 가져와 봅시다. 레벨 순서가 **`세부 → 성별 → 측정변수`** 인 점에 주의!

In [ ]:
# 세부 '전체' → 성별 '남학생' → 측정변수 '인지율'
boys_rate = stress['전체']['남학생']['인지율']
print(boys_rate)

In [ ]:
# 한 번에 접근하기 — xs (cross-section)
boys_rate_xs = stress.xs(('전체', '남학생', '인지율'), axis=1)
print(boys_rate_xs)

In [ ]:
# 시각화 — 시점별 남/여 스트레스 인지율
import matplotlib.pyplot as plt

girls_rate = stress['전체']['여학생']['인지율']

plt.figure(figsize=(8, 4))
plt.plot(boys_rate.index, boys_rate.values, 'o-', label='Boys')
plt.plot(girls_rate.index, girls_rate.values, 's-', label='Girls')
plt.xlabel('Year')
plt.ylabel('Stress awareness rate (%)')
plt.title('Stress awareness rate by gender (Total students)')
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. 본인 데이터에 적용해보기 ✏️

본인이 받은 공공데이터에 적용해봅시다.

**체크리스트**:
1. 헤더가 몇 행인지 확인 (head로 보고 어디까지가 헤더인지 판단)
2. `header=[0, 1, ...]` 옵션으로 읽기 (가장 빠른 방법)
3. `head()`로 **레벨 순서**(성별·측정변수 등) 확인 — 파일마다 다름!
4. 거슬리는 컬럼명은 §3처럼 `rename`으로 정리
5. 멀티인덱스 접근 시 `df['키1']['키2']` 또는 `df.xs()`

In [ ]:
# 본인 데이터로 시도해보기 (예시: 우울감_경험률.csv)
# 같은 KOSIS 구조라서 똑같이 적용 가능

depress = pd.read_csv('data/우울감_경험률.csv', header=[0, 1, 2, 3], index_col=0)
print(f"shape: {depress.shape}")

# 접근 전에 레벨 순서부터 확인! (파일마다 성별/측정변수 순서가 다를 수 있음)
print("컬럼 (앞 4개):")
for col in depress.columns[:4]:
    print(f"  {col}")
depress.head(3).iloc[:, :6]

In [ ]:
# 우울감 - 여학생 데이터 가져오기
# 위 head 출력에서 확인한 레벨 순서대로! (분류, 세부, 성별, 측정변수)
# 단위 박힌 컬럼명 그대로 사용 (정리하려면 §3의 rename 파이프라인 재사용)
depress_girls = depress[('전체', '소계', '여학생', '경험률 (%)')]
print(depress_girls.head())

## 6. ⚠️ 함정 / 주의사항

### 6.1 헤더 행 수 잘못 세기
헤더가 4행인데 `header=[0,1,2]` 처럼 3행만 주면 → 마지막 헤더가 첫 데이터 행이 됨.  
**해결**: raw CSV를 메모장이나 head 5줄로 확인해서 헤더 끝 정확히 파악.

### 6.2 레벨 순서 넘겨짚기
`df['전체']['인지율']['남학생']`처럼 머릿속 순서로 접근했는데, 실제 파일은 `성별 → 측정변수` 순서라 KeyError.  
**해결**: 접근 전에 `df.columns[:4]`로 **실제 튜플 순서를 눈으로 확인**. (이 노트북 데이터도 똑같은 함정이 있었어요!)

### 6.3 merge할 때 인덱스 안 맞음
멀티인덱스끼리 merge는 까다로움.  
**해결**: `df.reset_index()` 후 merge, 또는 단순 컬럼으로 정리한 다음 merge.

### 6.4 컬럼명 공백 / 단위 차이
`'인지율 (%)'` vs `'인지율'`처럼 공백 한 칸 차이로 KeyError.  
**해결**: 단위 제거할지 결정하고 일관되게 처리 (§3의 `rename` 후처리 활용).

## 7. 📚 더 알아보기

다음 학기 또는 자율 학습 주제:

- **`pd.pivot_table`** — groupby 결과를 자동으로 멀티인덱스로
- **`stack` / `unstack`** — 멀티인덱스를 행/열 사이로 옮기기
- **`pd.IndexSlice`** — 멀티인덱스 슬라이싱
- **`MultiIndex.from_product`** — 카르테시안 곱으로 만들기